In [2]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
import pandas as pd

In [3]:
from pathlib import Path

In [4]:
DATA_ROOT=Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")
path=DATA_ROOT/"simulated/shendure_pow_analysis"
name="sim_with_orthos_20251206"

In [5]:
spread_hypothesis_20251119=scm.HypothesisSet.from_tsv(path/"spread_hypothesis_20251119.tsv")

In [7]:
from dask.distributed import get_client#Semaphore, as_completed,

In [8]:
ortho_root=path/name/"orthos_with_precomputed_wald"
output_root=path/name/"results"
output_root.mkdir(exist_ok=True,parents=True)
input_ortho_names=[path.name for path in ortho_root.iterdir()]

#Semaphore(max_leases=10, name="test")

def compute_one_wald(input_root, name, output_root, hypothesis_set, hypothesis_set_name, test_type):
    #sem = Semaphore(name="test")

    client=get_client()
    ortho_oi=scm.ortho.load(client=client,
                                path=input_root,
                                name=name)
    scmpradat_oi=scm.scMPRA_data.from_parquet(path/"sim_with_orthos_20251206"/"scMPRA"/f"{name}.scmpra")
    ortho_oi.training_data=scmpradat_oi
    runner = scm.HypothesisTester(test_type)
    output_short=Path(output_root)/hypothesis_set_name/test_type
    output_short.mkdir(exist_ok=True,parents=True)
    #FOR WALD
    runner.run(hypothesis_set, ortho_oi, client).to_tsv(output_short/name)
    #FOR MWU
    #runner.run(hypothesis_set, scmpradat_oi, client).to_tsv(output_short/name)

In [9]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster=SLURMCluster(
        cores=4,#cores per slurm job
        memory="64G",#memory per slurm job
        processes=1,#dask workers per slurm job
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=2:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=1)
    client = Client(cluster,
            timeout=f"{10*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s",  # Worker heartbeat interval,
        )

In [10]:
futures_wald = [client.submit(compute_one_test,
                       input_root=ortho_root,
                       name=name_oi,
                       output_root=output_root,
                       hypothesis_set=ct_spread_hypothesis_20251119,
                       hypothesis_set_name="spread_hypothesis_20251119",
                       test_type="wald") for name_oi in input_ortho_names]

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_nb_versus_means', 'annotate_models', 'by_cell_type', 'by_cell_type_design', 'by_cell_type_parameters', 'by_cre', 'by_cre_design', 'by_cre_parameters', 'clean', 'compute_model_qc', 'criss_cross', 'extract_params', 'load', 'make_wald_eval_bundle', 'precompute_wald', 'save', 'training_data', 'wald_precomp']
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_

In [31]:
futures_wald

[<Future: finished, type: NoneType, key: compute_one_test-5e3e763195cdd6a771cacfe64fd674bb>,
 <Future: pending, key: compute_one_test-69d9901342520e0975affb32efe0a680>,
 <Future: pending, key: compute_one_test-02da806b789aac25550480312ec1bbd1>,
 <Future: pending, key: compute_one_test-2ec3cdc8ee6e655e89e553e815ba57d1>,
 <Future: pending, key: compute_one_test-e2d7e557a6bce814c7b2c192d502a760>]

In [ ]:
futures_mwu = [client.submit(compute_one_test,
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=spread_hypothesis_20251119,
                        hypothesis_set_name="spread_hypothesis_20251119",
                        test_type="mwu", use_client=True) for name_oi in input_ortho_names]

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_nb_versus_means', 'annotate_models', 'by_cell_type', 'by_cell_type_design', 'by_cell_type_parameters', 'by_cre', 'by_cre_design', 'by_cre_parameters', 'clean', 'compute_model_qc', 'criss_cross', 'extract_params', 'load', 'make_wald_eval_bundle', 'precompute_wald', 'save', 'training_data', 'wald_precomp']
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_

In [ ]:
results = [compute_one_test(
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=spread_hypothesis_20251119,
                        hypothesis_set_name="spread_hypothesis_20251119",
                        test_type="mwu") for name_oi in input_ortho_names]

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_nb_versus_means', 'annotate_models', 'by_cell_type', 'by_cell_type_design', 'by_cell_type_parameters', 'by_cre', 'by_cre_design', 'by_cre_parameters', 'clean', 'compute_model_qc', 'criss_cross', 'extract_params', 'load', 'make_wald_eval_bundle', 'precompute_wald', 'save', 'training_data', 'wald_precomp']
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_

In [ ]:
futures_mwu[0].result()

In [ ]:
client.dashboard_link

In [10]:
client.close()
cluster.close()

2025-12-08 17:14:27,062 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('compute_one_test-cfabaed90a31ded0f3060faafa8eac4b')" coro=<Worker.execute() done, defined at /home/eng26/.conda/envs/scmpra/lib/python3.10/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2025-12-08 17:14:27,062 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('compute_one_test-dc6f04e2d4c1e5e26510aff7ee081df3')" coro=<Worker.execute() done, defined at /home/eng26/.conda/envs/scmpra/lib/python3.10/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2025-12-08 17:14:27,063 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('compute_one_test-db8fc46e2b9ba7561fb2352328990c52')" coro=<Worker.execute() done, defined at /home/eng26/.conda/envs/scmpra/lib/python3.10/site-packages/distributed/worker_state_machin